In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import dotenv_values, load_dotenv
import plotly.io as pio

from suse.solar.nrel_api import download_solar_data
from suse.solar.solar_calc import compute_environmental_params, compute_panel_radiation
from suse.solar.panel_calc import compute_panel_e

pio.templates.default = "simple_white"
pd.options.plotting.backend = "plotly"

load_dotenv()
# NREL_KEY = dotenv_values("../.env")['NREL_API_KEY']
NREL_KEY = os.environ.get("NREL_API_KEY")
EMAIL = os.environ.get("EMAIL")

DATA_DIR = "../data/solar"

In [ ]:
# Input options
lat = 34.125448
lon = -118.038843
year = 2022
# solar_file = f'{DATA_DIR}/solar_data.parquet'

slope = lat  # or 35
grd_reflect = 0.2

In [155]:
geometry = f"POINT ({lon} {lat})"
df = download_solar_data(year=year, wkt=geometry, email=EMAIL, api_key=NREL_KEY, show_url=False)
df = compute_environmental_params(df, lon=lon, lat=lat)
df = compute_panel_radiation(df, lat=lat, slope=slope, grd_reflect=df["Surface Albedo"])
df = compute_panel_e(
    df,
    PV_n_panels=16,
    NOCT=45.7,
    beta=0.0045,
    module_rated_W=250,
    module_area=1.66,
    grid_price=0.125,
    install_cost_per_W=1.3
)

Total Monthly Radiation:
Fixed Panel: 2263964 Wh/m2

System rated power: 4.0 kW
Total install cost: $5200.0
Total PV Electricity: 7425 kWh/yr
Total Energy Savings: $928.12 
Avg Efficiency: 0.135


In [ ]:
# geometry = "POLYGON ((-118.262329 33.628342, -117.751465 33.628342, -117.751465 34.077687, -118.262329 34.077687, -118.262329 33.628342))"
try:
    raise ValueError
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    df = pd.read_parquet(solar_file)
except:
    df.to_parquet(solar_file)

In [ ]:
df["PV_W_e"].sum()

np.float64(7424955.250234912)

In [ ]:
df.plot(y=["PV_n_c", "PV_U_L", "PV_T_c", "PV_n", "PV_W_e"])

In [ ]:
df.plot(y="G_T_fixed")

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_traces(go.Scatter(x=df["Datetime"], y=df["PV_W_e"], name="PV Electricity"))
fig.update_traces(name="PV Electricity", showlegend=True)
fig.add_trace(go.Scatter(x=df["Datetime"], y=df["PV_n"], name="Efficiency"), secondary_y=True)
fig.update_layout(
    yaxis_title="Electricity [W]",
    yaxis2_title="Efficiency",
)

fig.show("vscode")

In [ ]:
df.plot(x="Datetime", y=["GHI (W/m^2)", "DNI (W/m^2)", "G_b", "G_d"])

In [ ]:
# https://www.nrel.gov/docs/fy25osti/92257.pdf

# Page 28 pricing for industry